  04 — HQ200 car-only reference mesh cleaning

The supplied scanner mesh contains the car and background geometry. This notebook lets you inspect every scene,
choose an explicit 3D crop, preview it, and approve it before export. Raw OBJ files are never modified.

Output: `data_processed/reference_meshes/3DRealCar/<scene>/car_reference.obj`.
Because automatic foreground extraction from a connected room/car mesh is unreliable, every exported crop requires
manual approval. The saved JSON makes the process reproducible.


In [2]:
%pip -q install trimesh scipy pandas matplotlib ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.5 MB/s eta 0:00:00a 0:00:01


In [3]:
import gc, json, shutil, subprocess, sys
from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = Path("/content/drive/MyDrive/ITU/3D/Thesis")
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "ht" + "tps:" + chr(47)*2 + "github.com" + chr(47) + "katlit" + chr(47) + "Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if CODE_ROOT.exists() and not (CODE_ROOT / ".git").is_dir(): shutil.rmtree(CODE_ROOT)
command = (["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)]
           if not CODE_ROOT.exists() else ["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH])
subprocess.run(command, check=True)
sys.path.insert(0, str(CODE_ROOT / "code"))

from src.mesh_cleanup import crop_mesh, export_clean_reference, load_triangle_mesh, mesh_summary, sample_for_display


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def find_unique_dir(names, roots):
    matches = [root / name for root in roots for name in names if (root / name).is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one HQ200 root, found: {matches}")
    return matches[0]

HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], [PROJECT_ROOT / "data", PROJECT_ROOT])
if (HQ200_ROOT / "3DrealCarHQ200").is_dir(): HQ200_ROOT /= "3DrealCarHQ200"
OUTPUT_ROOT = PROJECT_ROOT / "data_processed/reference_meshes/3DRealCar"
CONFIG_PATH = PROJECT_ROOT / "splits/reference_mesh_crop_boxes.json"
mesh_paths = {path.parent.name: path for path in HQ200_ROOT.glob("*/textured_output.obj")}
if not mesh_paths: raise FileNotFoundError(f"No textured_output.obj files below {HQ200_ROOT}")
scenes = sorted(mesh_paths)
rows = []
for scene, path in mesh_paths.items():
    inspected_mesh = load_triangle_mesh(path)
    rows.append(mesh_summary(inspected_mesh, scene, path))
    del inspected_mesh
    gc.collect()
display(pd.DataFrame(rows).sort_values("scene").round(3))
print("Scenes:", len(scenes), "| raw meshes are read-only")


,scene,path,vertices,faces,x_min,x_max,x_extent,y_min,y_max,y_extent,z_min,z_max,z_extent
2,2024_09_11_12_52_22,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,194879,317250,-3.950,4.490,8.440,-0.738,0.899,1.637,-4.190,4.170,8.360
4,2024_09_12_16_22_02,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,111460,191470,-4.095,3.913,8.008,-0.580,0.871,1.451,-3.731,3.731,7.462
9,2024_09_21_10_56_00_anonymous _SUV,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,103571,180464,-3.237,3.601,6.838,-0.812,0.949,1.761,-3.991,3.965,7.956
0,2024_09_21_11_09_11_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,116759,198084,-3.055,4.043,7.098,-0.681,0.819,1.500,-4.173,4.459,8.632
1,2024_09_24_09_27_07_anonymous_suv,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,102485,181478,-3.497,3.445,6.942,-0.758,0.923,1.681,-3.913,3.601,7.514
7,2024_09_25_15_13_47_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,96723,161743,-3.185,3.601,6.786,-0.513,1.001,1.514,-3.887,3.887,7.774
8,2024_09_26_09_25_50_anonymous _suv,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,96136,166495,-3.289,3.965,7.254,-0.885,1.079,1.964,-3.315,3.471,6.786
5,2024_09_26_09_52_41_anonymous _suv,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,96988,167369,-3.289,3.627,6.916,-0.799,0.871,1.670,-3.835,3.653,7.488
3,2024_09_27_11_24_21_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,94725,163655,-3.445,3.731,7.176,-0.741,0.845,1.586,-3.393,3.731,7.124
6,2024_09_27_14_55_26_anonymous _mpv,/content/drive/MyDrive/ITU/3D/Thesis/data/3Dre...,125649,200812,-3.679,3.654,7.332,-1.864,1.906,3.771,-3.564,3.553,7.117


Scenes: 10 | raw meshes are read-only


   Interactive crop and approval

The sliders are normalized to each mesh's complete bounds: `0` is its minimum and `1` its maximum on that axis.
Adjust the three ranges until the right-hand preview contains the complete car but no surrounding scan. Check at
least two rotations mentally by rerunning the preview; the plot shows two automatically. Click **Approve and save**
only when the complete car is retained. Saving the crop does not yet export a mesh.


In [5]:
saved = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else {}
default_box = [[0.25, 0.75], [0.05, 0.95], [0.25, 0.75]]
scene_widget = widgets.Dropdown(options=scenes, description="Scene:", layout=widgets.Layout(width="750px"))
sliders = [widgets.FloatRangeSlider(value=default_box[i], min=0, max=1, step=.01,
           description=axis, continuous_update=False, layout=widgets.Layout(width="700px"))
           for i, axis in enumerate(["X", "Y", "Z"])]
preview_button = widgets.Button(description="Preview crop", button_style="info")
approve_button = widgets.Button(description="Approve and save", button_style="success")
output = widgets.Output()

def current_box(): return [list(slider.value) for slider in sliders]

def load_saved_box(change=None):
    box = saved.get(scene_widget.value, {}).get("normalized_box", default_box)
    for slider, limits in zip(sliders, box): slider.value = tuple(limits)

def show_preview(_=None):
    with output:
        clear_output(wait=True)
        scene = scene_widget.value
        original = load_triangle_mesh(mesh_paths[scene])
        try: cropped = crop_mesh(original, current_box())
        except Exception as error:
            print("Invalid crop:", error); return
        original_points = sample_for_display(original, 5_000)
        cropped_points = sample_for_display(cropped, 8_000)
        fig = plt.figure(figsize=(18, 8))
        for index, (points, title, azimuth) in enumerate([
            (original_points, "Original mesh including background", -65),
            (cropped_points, "Proposed car-only crop", -65),
            (cropped_points, "Proposed crop — second angle", 25),
        ], start=1):
            axis = fig.add_subplot(1, 3, index, projection="3d")
            axis.scatter(*points.T, s=.15, c="0.25")
            axis.set_title(title); axis.set_box_aspect(np.ptp(points, axis=0).clip(min=1e-6))
            axis.view_init(18, azimuth)
        plt.tight_layout(); plt.show(); plt.close(fig)
        display(pd.DataFrame([mesh_summary(cropped, scene)]).round(3))
        del original, cropped, original_points, cropped_points
        gc.collect()

def approve(_):
    scene = scene_widget.value
    original = load_triangle_mesh(mesh_paths[scene])
    cropped = crop_mesh(original, current_box())
    saved[scene] = {"normalized_box": current_box(), "approved": True,
                    "raw_mesh": str(mesh_paths[scene]), "cropped_summary": mesh_summary(cropped, scene)}
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(json.dumps(saved, indent=2))
    with output: print("Approved and saved:", scene, "→", CONFIG_PATH)
    del original, cropped
    gc.collect()

scene_widget.observe(load_saved_box, names="value")
preview_button.on_click(show_preview); approve_button.on_click(approve)
load_saved_box()
display(widgets.VBox([scene_widget, *sliders, widgets.HBox([preview_button, approve_button]), output]))
show_preview()


   Approval status and guarded batch export


In [8]:
saved = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else {}
status = pd.DataFrame([{"scene": scene, "approved": bool(saved.get(scene, {}).get("approved", False)),
                        "already_exported": (OUTPUT_ROOT / scene / "car_reference.obj").is_file()}
                       for scene in scenes])
display(status)
print("Approved:", int(status.approved.sum()), "/", len(status))


,scene,approved,already_exported
0,2024_09_11_12_52_22,True,False
1,2024_09_12_16_22_02,True,False
2,2024_09_21_10_56_00_anonymous _SUV,True,False
3,2024_09_21_11_09_11_anonymous _sedan,True,False
4,2024_09_24_09_27_07_anonymous_suv,True,False
5,2024_09_25_15_13_47_anonymous _sedan,True,False
6,2024_09_26_09_25_50_anonymous _suv,True,False
7,2024_09_26_09_52_41_anonymous _suv,True,False
8,2024_09_27_11_24_21_anonymous _sedan,True,False
9,2024_09_27_14_55_26_anonymous _mpv,True,False


Approved: 10 / 10


In [9]:
RUN_EXPORT = True
OVERWRITE = False

if RUN_EXPORT:
    unapproved = [scene for scene in scenes if not saved.get(scene, {}).get("approved", False)]
    if unapproved:
        raise RuntimeError(f"Approve every scene before batch export. Missing: {unapproved}")
    export_rows = []
    for scene in scenes:
        original = load_triangle_mesh(mesh_paths[scene])
        cleaned = crop_mesh(original, saved[scene]["normalized_box"])
        destination = OUTPUT_ROOT / scene / "car_reference.obj"
        export_clean_reference(cleaned, destination, overwrite=OVERWRITE)
        export_rows.append(mesh_summary(cleaned, scene, destination))
        del original, cleaned
        gc.collect()
    manifest = pd.DataFrame(export_rows)
    manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
    display(manifest.round(3)); print("Saved:", OUTPUT_ROOT)
else:
    print("Dry run. After approving every scene, set RUN_EXPORT=True.")


,scene,path,vertices,faces,x_min,x_max,x_extent,y_min,y_max,y_extent,z_min,z_max,z_extent
0,2024_09_11_12_52_22,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,46856,77162,-0.853,1.210,2.063,-0.660,0.830,1.490,-1.369,1.420,2.789
1,2024_09_12_16_22_02,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,24565,45189,-1.229,1.125,2.354,-0.325,0.714,1.040,-2.332,2.652,4.984
2,2024_09_21_10_56_00_anonymous _SUV,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,38677,68999,-1.539,1.898,3.437,-0.611,0.871,1.482,-2.015,2.662,4.677
3,2024_09_21_11_09_11_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,24264,43799,-2.847,1.164,4.011,-0.429,0.498,0.927,-2.724,1.925,4.650
4,2024_09_24_09_27_07_anonymous_suv,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,50221,89352,-1.805,1.722,3.528,-0.689,0.845,1.534,-2.041,1.729,3.770
5,2024_09_25_15_13_47_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,46154,78136,-1.521,1.911,3.432,-0.420,0.930,1.349,-1.964,1.956,3.920
6,2024_09_26_09_25_50_anonymous _suv,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,40362,71394,-1.482,2.171,3.653,-0.771,0.992,1.763,-1.631,1.781,3.412
7,2024_09_26_09_52_41_anonymous _suv,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,33072,58248,-1.592,1.911,3.503,-0.720,0.793,1.513,-1.979,1.781,3.760
8,2024_09_27_11_24_21_anonymous _sedan,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,46925,81795,-1.677,1.937,3.614,-0.653,0.769,1.422,-1.626,1.963,3.589
9,2024_09_27_14_55_26_anonymous _mpv,/content/drive/MyDrive/ITU/3D/Thesis/data_proc...,49662,78730,-1.865,1.843,3.708,-1.694,1.726,3.420,-1.807,1.796,3.603


Saved: /content/drive/MyDrive/ITU/3D/Thesis/data_processed/reference_meshes/3DRealCar
